# CV8 – Překlad slov pomocí vektorových reprezentací

Cílem cvičení je natrénovat lineární transformační matici $W$, která mapuje embeddingy slov ze zdrojového jazyka (čeština) do prostoru embeddingů cílového jazyka (angličtina). Využijeme předtrénované fastText vektory a překladový slovník MUSE.

## Importy

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os


---
## Otázka: Co to je embedding?

**Embedding** (vektorová reprezentace) je mapování diskrétních objektů (např. slov) do spojitého vektorového prostoru $\mathbb{R}^d$, kde $d$ je dimenze embeddingu.

Klíčové vlastnosti:
- Každé slovo je reprezentováno hustým vektorem reálných čísel (např. 300 dimenzí u fastText).
- Sémanticky nebo syntakticky podobná slova mají v tomto prostoru blízké vektory (měřeno např. kosinovou podobností).
- Embeddingy zachycují analogie: klasický příklad je $\vec{\text{král}} - \vec{\text{muž}} + \vec{\text{žena}} \approx \vec{\text{královna}}$.
- FastText rozšiřuje Word2Vec tím, že pracuje se subword n-gramy — dokáže generovat vektory i pro slova, která nebyla v trénovacím korpusu.

Embeddingy se trénují tak, aby slova vyskytující se v podobných kontextech měla podobné vektory (distributional hypothesis).

---
## Část 1 – Příprava dat (1 bod)

### Data

Všechny potřebné soubory jsou uloženy ve stejné složce jako tento notebook:

- `cc.cs.300.vec` — fastText vektory pro češtinu (textový formát)
- `cc.en.300.vec` — fastText vektory pro angličtinu (textový formát)
- `cs-en.0-5000.txt` — trénovací překladový slovník MUSE
- `cs-en.5000-6500.txt` — testovací překladový slovník MUSE

**Poznámka:** Formát `.vec` neobsahuje subword model — vektory jsou dostupné pouze
pro slova přímo ve slovníku. Páry s OOV slovy jsou při sestavování matic přeskočeny.


In [ ]:
REQUIRED_FILES = [
    'cc.cs.300.vec',
    'cc.en.300.vec',
    'cs-en.0-5000.txt',
    'cs-en.5000-6500.txt',
]

for filename in REQUIRED_FILES:
    if os.path.exists(filename):
        print(f'Nalezen: {filename}')
    else:
        raise FileNotFoundError(f'Chybí soubor: {filename}')


In [2]:
def load_vec_model(lang_code, max_words=200000):
    """
    Načte fastText vektory z textového souboru .vec.
    Formát: první řádek = "vocab_size dim", každý další = "slovo f1 f2 ... fd".

    Vrátí:
        word2vec : dict {slovo -> np.array(dim)}
        words    : list slov (v pořadí ze souboru)
        matrix   : np.array tvaru (n_slov, dim)
    """
    model_path = f'cc.{lang_code}.300.vec'

    if not os.path.exists(model_path):
        raise FileNotFoundError(f'Chybí soubor: {model_path}')

    print(f'Načítám model {model_path}...')
    words = []
    vectors = []

    with open(model_path, 'r', encoding='utf-8') as f:
        vocab_size, dim = map(int, f.readline().strip().split())
        for i, line in enumerate(f):
            if i >= max_words:
                break
            parts = line.rstrip().split(' ')
            words.append(parts[0])
            vectors.append(np.array(parts[1:], dtype=np.float32))

    matrix = np.array(vectors, dtype=np.float32)
    word2vec = {w: matrix[i] for i, w in enumerate(words)}

    print(f'Hotovo: načteno {len(words)} slov, matice tvaru {matrix.shape}')
    return word2vec, words, matrix


In [3]:
def load_dictionary(filepath):
    """
    Načte MUSE překladový slovník.
    Formát: každý řádek obsahuje 'slovo_cs\tslovo_en' nebo 'slovo_cs slovo_en'.

    Vrátí:
        pairs : list dvojic (src_word, tgt_word)
    """
    pairs = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Podpora tabulátoru i mezery jako oddělovače
            if '\t' in line:
                parts = line.split('\t', 1)
            else:
                parts = line.split(' ', 1)
            if len(parts) == 2:
                pairs.append((parts[0].strip(), parts[1].strip()))
    print(f"Načteno {len(pairs)} překladových dvojic ze souboru {os.path.basename(filepath)}")
    return pairs

In [4]:
def build_matrices(src_word2vec, tgt_word2vec, pairs):
    """
    Sestaví trénovací matice X (zdrojové embeddingy) a Y (cílové embeddingy)
    pro překladové dvojice. Páry, kde jedno ze slov chybí ve slovníku, jsou
    přeskočeny (formát .vec nepodporuje generování vektorů pro OOV slova).

    Parametry:
        src_word2vec : dict {slovo -> np.array} (zdrojový jazyk)
        tgt_word2vec : dict {slovo -> np.array} (cílový jazyk)
        pairs        : list dvojic (src_word, tgt_word)

    Vrátí:
        X : np.array tvaru (n_pairs, src_dim)
        Y : np.array tvaru (n_pairs, tgt_dim)
    """
    X_rows = []
    Y_rows = []
    skipped = 0

    for src_word, tgt_word in pairs:
        if src_word not in src_word2vec or tgt_word not in tgt_word2vec:
            skipped += 1
            continue
        X_rows.append(src_word2vec[src_word])
        Y_rows.append(tgt_word2vec[tgt_word])

    if skipped > 0:
        print(f'Přeskočeno {skipped} dvojic (OOV slova)')

    if len(X_rows) == 0:
        raise ValueError(
            "Žádné platné dvojice nenalezeny — zkontrolujte slovníky."
        )

    X = np.array(X_rows, dtype=np.float32)
    Y = np.array(Y_rows, dtype=np.float32)
    print(f'Sestaveny matice: X={X.shape}, Y={Y.shape}')
    return X, Y


In [5]:
TRAIN_DICT_PATH = 'cs-en.0-5000.txt'
TEST_DICT_PATH  = 'cs-en.5000-6500.txt'

print("=== Načítání CZ vektorů ===")
cs_word2vec, cs_words, cs_matrix = load_vec_model('cs', max_words=200000)

print("\n=== Načítání EN vektorů ===")
en_word2vec, en_words, en_matrix = load_vec_model('en', max_words=200000)

print("\n=== Načítání slovníků ===")
train_pairs = load_dictionary(TRAIN_DICT_PATH)
test_pairs  = load_dictionary(TEST_DICT_PATH)

print("\n=== Sestavování trénovacích matic ===")
X_train, Y_train = build_matrices(cs_word2vec, en_word2vec, train_pairs)

print("\n=== Sestavování testovacích matic ===")
X_test, Y_test = build_matrices(cs_word2vec, en_word2vec, test_pairs)


---
## Otázka: Co dělá gradient descent?

**Gradient descent** (metoda gradientního sestupu) je iterativní optimalizační algoritmus pro minimalizaci účelové funkce (loss funkce).

Princip:
1. Začneme s počátečními hodnotami parametrů (např. náhodnou maticí $W^T$).
2. Vypočítáme **gradient** $\nabla_{W^T} L$ — vektor (resp. matici) parciálních derivací loss funkce $L$ podle všech parametrů. Gradient ukazuje směr **nejstrmějšího růstu** $L$.
3. Parametry aktualizujeme v **opačném směru** gradientu (tedy ve směru sestupu):
   $$W^T \leftarrow W^T - \alpha \cdot \nabla_{W^T} L$$
   kde $\alpha > 0$ je **learning rate** (krok učení).
4. Opakujeme, dokud loss neklesne pod požadovanou mez nebo nedosáhneme maximálního počtu kroků.

Intuice: představte si, že stojíte na kopci v mlze a chcete sejít do údolí. V každém kroku se podíváte pod nohy (gradient) a uděláte krok dolů po svahu.

Volba $\alpha$:
- Příliš velké $\alpha$ → oscilace, divergence.
- Příliš malé $\alpha$ → pomalá konvergence.
- Pro tento problém (s fastText vektory dimenze 300) je vhodná malá hodnota, např. $\alpha \approx 10^{-4}$ až $10^{-2}$.

---
## Část 2 – Implementace trénovacího algoritmu (6 bodů)

Hledáme transformační matici $W$ takovou, aby platilo:
$$X W^T \approx Y$$
kde:
- $X \in \mathbb{R}^{n \times d_{\text{cs}}}$ jsou embeddingy zdrojových slov (čeština),
- $Y \in \mathbb{R}^{n \times d_{\text{en}}}$ jsou embeddingy cílových slov (angličtina),
- $W^T \in \mathbb{R}^{d_{\text{cs}} \times d_{\text{en}}}$ je hledaná transformační matice.

### Loss funkce

Minimalizujeme **čtvercovou Frobeniovu normu** residuální matice:
$$L(W^T) = \|X W^T - Y\|_F^2 = \sum_{i,j} (X W^T - Y)_{ij}^2$$

### Odvození gradientu

Označme $D = X W^T - Y$ (residuál). Pak:
$$L = \|D\|_F^2 = \mathrm{trace}(D^T D) = \sum_{i,j} D_{ij}^2$$

Gradient $L$ vůči $W^T$:
$$\frac{\partial L}{\partial (W^T)} = \frac{\partial}{\partial (W^T)} \|X W^T - Y\|_F^2$$

Použijeme pravidlo pro derivaci Frobeniovy normy: pokud $L = \|A Z - B\|_F^2$, pak $\frac{\partial L}{\partial Z} = 2 A^T (AZ - B)$.

V našem případě $A = X$, $Z = W^T$, $B = Y$:
$$\nabla_{W^T} L = 2 X^T (X W^T - Y) = 2 X^T D$$

Rozměry: $X^T \in \mathbb{R}^{d_{\text{cs}} \times n}$, $D \in \mathbb{R}^{n \times d_{\text{en}}}$ → gradient $\in \mathbb{R}^{d_{\text{cs}} \times d_{\text{en}}}$ — stejný tvar jako $W^T$. ✓

In [ ]:
def frobenius_norm_squared(X, W_T, Y):
    """Vypočítá ||XW^T - Y||_F^2"""
    D = X @ W_T - Y
    return np.sum(D ** 2)


def compute_residual(X, W_T, Y):
    """Vypočítá XW^T - Y"""
    return X @ W_T - Y


def compute_gradient(X, W_T, Y):
    """Vypočítá gradient L vůči W^T: 2 * X^T @ (X @ W^T - Y)"""
    D = compute_residual(X, W_T, Y)
    return 2 * X.T @ D


def gradient_descent_step(W_T, gradient, alpha):
    """Jeden krok gradient descent"""
    return W_T - alpha * gradient

In [ ]:
def train(X_train, Y_train, alpha=0.01, max_steps=1000, convergence_window=10, min_improvement=1e-6):
    """
    Trénování transformační matice pomocí gradient descent.

    Zastaví se buď po max_steps nebo při konvergenci
    (pokud v posledních convergence_window iteracích nedojde
    k poklesu loss o min_improvement).

    Parametry:
        X_train           : np.array (n, src_dim)
        Y_train           : np.array (n, tgt_dim)
        alpha             : learning rate
        max_steps         : maximální počet iterací
        convergence_window: počet posledních kroků pro detekci konvergence
        min_improvement   : minimální pokles loss pro pokračování

    Vrátí:
        W_T          : naučená transformační matice (src_dim, tgt_dim)
        loss_history : list hodnot loss v každém kroku
    """
    n_src = X_train.shape[1]  # src_dim
    n_tgt = Y_train.shape[1]  # tgt_dim

    # Inicializace W^T malými náhodnými hodnotami
    W_T = np.random.randn(n_src, n_tgt).astype(np.float32) * 0.01

    loss_history = []

    for step in range(max_steps):
        loss = frobenius_norm_squared(X_train, W_T, Y_train)
        loss_history.append(loss)

        # Detekce konvergence: porovnání s hodnotou před convergence_window kroky
        if len(loss_history) > convergence_window:
            recent_improvement = loss_history[-convergence_window - 1] - loss_history[-1]
            if recent_improvement < min_improvement:
                print(f"Konvergence dosažena v kroku {step}")
                break

        grad = compute_gradient(X_train, W_T, Y_train)
        W_T = gradient_descent_step(W_T, grad, alpha)

        if step % 100 == 0:
            print(f"Krok {step}: Loss = {loss:.4f}")

    return W_T, loss_history

In [ ]:
# Spuštění tréninku
# Learning rate je citlivý parametr — pro fastText vektory (dimenze 300) doporučujeme začít s 1e-4
np.random.seed(42)

W_T, loss_history = train(
    X_train,
    Y_train,
    alpha=1e-4,
    max_steps=500,
    convergence_window=10,
    min_improvement=1e-6
)

In [ ]:
# Vizualizace průběhu loss
plt.figure(figsize=(10, 4))
plt.plot(loss_history, color='steelblue', linewidth=1.5)
plt.xlabel('Krok (iterace)')
plt.ylabel('Loss $\\|XW^T - Y\\|_F^2$')
plt.title('Průběh loss funkce během gradient descent')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Počáteční loss: {loss_history[0]:.4f}")
print(f"Konečná loss:   {loss_history[-1]:.4f}")
print(f"Počet kroků:    {len(loss_history)}")

---
## Otázka: Jak bychom museli změnit matice X a Y, pokud bychom použili matici W místo $W^T$?

V našem modelu platí:
$$X W^T \approx Y$$
kde $W^T \in \mathbb{R}^{d_{\text{cs}} \times d_{\text{en}}}$, tedy $W \in \mathbb{R}^{d_{\text{en}} \times d_{\text{cs}}}$.

Pokud bychom chtěli použít $W$ **přímo** (bez transpozice), hledáme:
$$X W \approx Y$$
Pak $W$ musí mít tvar $(d_{\text{cs}} \times d_{\text{en}})$ — stejný jako $W^T$ výše. Jinými slovy, $W$ by bylo jednoduše tím, co jsme dosud nazývali $W^T$ — jde jen o přejmenování.

Pokud bychom naopak chtěli zapsat model jako:
$$W X^T \approx Y^T$$
pak $W \in \mathbb{R}^{d_{\text{en}} \times d_{\text{cs}}}$ a matice $X$, $Y$ by musely být **transponovány**: $X^T \in \mathbb{R}^{d_{\text{cs}} \times n}$, $Y^T \in \mathbb{R}^{d_{\text{en}} \times n}$.

**Závěr:** Použití $W$ místo $W^T$ samo o sobě **nevyžaduje změnu matic X a Y** — jde o ekvivalentní formulaci, kde $W = (W^T)^T$. Matice X a Y by zůstaly ve stejném tvaru $(n \times d_{\text{src}})$ resp. $(n \times d_{\text{tgt}})$, protože $X @ W \approx Y$ s $W \in \mathbb{R}^{d_{\text{cs}} \times d_{\text{en}}}$ je totéž co $X @ W^T \approx Y$ pokud $W$ v novém zápisu odpovídá původnímu $W^T$.

---
## Část 3 – Překlad a vyhodnocení (2 body)

In [ ]:
def translate_word(word, src_word2vec, W_T, tgt_matrix, tgt_words, top_k=5):
    """
    Přeloží slovo ze zdrojového do cílového jazyka.

    Parametry:
        word         : vstupní slovo ve zdrojovém jazyce
        src_word2vec : dict {slovo -> np.array} zdrojového jazyka
        W_T          : naučená transformační matice (src_dim, tgt_dim)
        tgt_matrix   : matice embeddingů cílového jazyka (vocab_size, tgt_dim)
        tgt_words    : list slov cílového jazyka (odpovídá řádkům tgt_matrix)
        top_k        : počet nejpodobnějších slov

    Vrátí:
        list dvojic (slovo, kosinová_podobnost), nebo [] pokud slovo není ve slovníku
    """
    if word not in src_word2vec:
        return []

    src_vec = src_word2vec[word].astype(np.float32)
    translated = src_vec @ W_T

    norms = np.linalg.norm(tgt_matrix, axis=1)
    translated_norm = np.linalg.norm(translated)

    if translated_norm == 0:
        return []

    similarities = tgt_matrix @ translated / (norms * translated_norm + 1e-10)
    top_indices = np.argsort(similarities)[-top_k:][::-1]

    return [(tgt_words[i], float(similarities[i])) for i in top_indices]


In [ ]:
def evaluate_accuracy(test_pairs, src_word2vec, W_T, tgt_matrix, tgt_words, top_k=5):
    """
    Vyhodnotí přesnost překladu na testovacích dvojicích.

    Vrátí:
        acc1  : accuracy@1 (správný překlad na 1. místě)
        acc5  : accuracy@5 (správný překlad v top 5)
        total : počet vyhodnocených dvojic
    """
    correct_top1 = 0
    correct_top5 = 0
    total = 0

    for src_word, expected_tgt in test_pairs:
        results = translate_word(src_word, src_word2vec, W_T, tgt_matrix, tgt_words, top_k=5)
        if not results:
            continue
        predicted_words = [w for w, _ in results]

        total += 1
        if predicted_words[0] == expected_tgt:
            correct_top1 += 1
        if expected_tgt in predicted_words:
            correct_top5 += 1

    acc1 = correct_top1 / total if total > 0 else 0.0
    acc5 = correct_top5 / total if total > 0 else 0.0

    return acc1, acc5, total


In [ ]:
demo_words = ['pes', 'kočka', 'dům', 'auto', 'voda', 'kniha', 'škola', 'Praha']

print("=== Ukázka překladu ===")
print(f"{'CZ slovo':<15} {'Top 5 anglických překladů (podobnost)'}")
print('-' * 70)

for word in demo_words:
    results = translate_word(word, cs_word2vec, W_T, en_matrix, en_words, top_k=5)
    if results:
        top5_str = ', '.join(f"{w} ({s:.3f})" for w, s in results)
        print(f"{word:<15} {top5_str}")
    else:
        print(f"{word:<15} (slovo není ve slovníku)")


In [ ]:
print("=== Vyhodnocení na testovací sadě ===")
acc1, acc5, total = evaluate_accuracy(
    test_pairs,
    cs_word2vec,
    W_T,
    en_matrix,
    en_words,
    top_k=5
)

print(f"Testovacích dvojic vyhodnoceno : {total}")
print(f"Accuracy@1                     : {acc1:.4f}  ({acc1*100:.2f} %)")
print(f"Accuracy@5                     : {acc5:.4f}  ({acc5*100:.2f} %)")


---
## Shrnutí

V tomto cvičení jsme:
1. Načetli předtrénované fastText embeddingy pro češtinu a angličtinu.
2. Z překladového slovníku MUSE sestavili trénovací páry $(x_i, y_i)$, kde $x_i$ jsou CZ embeddingy a $y_i$ jsou EN embeddingy.
3. Natrénovali lineární transformační matici $W^T$ pomocí gradient descent, minimalizací $\|XW^T - Y\|_F^2$.
4. Implementovali překlad pomocí kosinové podobnosti v cílovém prostoru.
5. Vyhodnotili accuracy@1 a accuracy@5 na testovací sadě.

Metoda je známá jako **MUSE (Multilingual Unsupervised and Supervised Embeddings)** v supervised variantě — lineární mapování mezi jazykovými prostory.